# TrustLoop × Octo-Small 1.5 — standalone inference (Colab)

Runs the **pretrained** `rail-berkeley/octo-small-1.5` model (no fine-tuning) on MuJoCo RGB screenshots and saves the predicted action chunks as JSON.

This notebook does **not** control any robot. Octo's actions follow the convention of a training dataset (e.g. Bridge/WidowX) and are **not** TrustLoop commands.

**Before running:** Runtime → Change runtime type → **T4 GPU**. CPU also works, just slower.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "No GPU - Octo will run on CPU"
!python3 --version

## 1. Upload the two code files
Upload `octo_inference.py`, `setup_octo_colab.sh` and `octo_constraints.txt` from the `octo_colab/` folder.

In [ ]:
import os
os.makedirs('/content/trustloop_octo', exist_ok=True)
%cd /content/trustloop_octo
from google.colab import files
uploaded = files.upload()          # select octo_inference.py, setup_octo_colab.sh, octo_constraints.txt
print(sorted(os.listdir('.')))

## 2. Build the isolated Octo environment (≈5–10 min, once per runtime)
Octo pins TensorFlow 2.15 / JAX 0.4.20, which don't install on Colab's own Python. The script creates a Python 3.10 venv at `/content/octo_env/venv` and leaves Colab's packages untouched.

In [ ]:
!bash setup_octo_colab.sh
OCTO_PY = '/content/octo_env/venv/bin/python'

## 3. Load the model and list its dataset statistics
The first run downloads the checkpoint from Hugging Face. The listed keys are the datasets whose action conventions can be used to unnormalise outputs.

In [ ]:
!{OCTO_PY} octo_inference.py --list-stats

## 4. Upload MuJoCo screenshots
For example, `logs/screenshots/*_home.png` from TrustLoop (`python src/main.py --headless --screenshots`). Any RGB PNG or JPG works; it is resized to the checkpoint's `image_primary` size.

In [ ]:
os.makedirs('/content/trustloop_octo/images', exist_ok=True)
%cd /content/trustloop_octo/images
imgs = files.upload()
%cd /content/trustloop_octo
IMAGES = sorted(os.path.join('images', k) for k in imgs)
print(IMAGES)

## 5. Self-check (shapes, normalisation, determinism)
Checks the output shape against the model config, re-derives unnormalisation by hand from the checkpoint's mean/std/mask, and confirms the same seed gives the same result. Exit code 0 means all checks passed.

In [ ]:
INSTRUCTION = "put the red cube in the blue bin"
!{OCTO_PY} octo_inference.py --image "{IMAGES[0]}" --instruction "{INSTRUCTION}" --self-check --out self_check.json

## 6. Run inference on every screenshot
Single-frame window (history 1). To use Octo's pretraining history of 2, add `--prev-image <earlier frame>`.

In [ ]:
import json, subprocess
os.makedirs('results', exist_ok=True)
for img in IMAGES:
    out = os.path.join('results', os.path.splitext(os.path.basename(img))[0] + '_octo.json')
    r = subprocess.run([OCTO_PY, 'octo_inference.py', '--image', img, '--instruction', INSTRUCTION,
                        '--stats-key', 'bridge_dataset', '--samples', '1', '--seed', '0', '--out', out],
                       capture_output=True, text=True)
    print(r.stdout[-1500:]); print(r.stderr[-800:] if r.returncode else '')

## 7. Look at the predictions
Rows are the steps in the action chunk; columns are action dimensions **in the training dataset's convention**. For `bridge_dataset` (EEF_POS encoding in Octo's dataset config) these are Δx, Δy, Δz, Δroll, Δpitch, Δyaw and gripper; the gripper dimension is not unnormalised (mask = False).

In [ ]:
import glob, numpy as np, matplotlib.pyplot as plt
from PIL import Image
for path in sorted(glob.glob('results/*_octo.json')):
    r = json.load(open(path))
    a = np.array(r['actions'])
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={'width_ratios': [1, 2]})
    ax0.imshow(Image.open(r['image'])); ax0.set_title(os.path.basename(r['image'])); ax0.axis('off')
    for d in range(a.shape[1]):
        ax1.plot(a[:, d], marker='o', label=f'dim {d}' + (' (not unnormalised)' if not r['normalization_mask'][d] else ''))
    ax1.set_xlabel('step in action chunk'); ax1.set_title(f"{r['stats_key']} | {r['instruction']}")
    ax1.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()
    print(np.array2string(a, precision=4, suppress_small=True))

## 8. Download the results

In [ ]:
!cd /content/trustloop_octo && zip -qr octo_results.zip results self_check.json
files.download('/content/trustloop_octo/octo_results.zip')

## 9. TrustLoop shadow mode: batch inference on an exported manifest
On the laptop:
```
python src/experiments.py --experiments octo_shadow --trials 5 --out logs/experiments/shadow
cd logs/experiments/shadow && zip -qr octo_shadow.zip octo_shadow
```
Upload `octo_shadow.zip` here. The model is loaded **once** for all frames, and re-running resumes where it stopped.

In [ ]:
%cd /content/trustloop_octo
up = files.upload()                      # octo_shadow.zip
!rm -rf octo_shadow && unzip -q octo_shadow.zip && ls octo_shadow | head
!{OCTO_PY} octo_inference.py --manifest octo_shadow/manifest.jsonl --use-history --stats-key bridge_dataset --seed 0

In [ ]:
files.download('/content/trustloop_octo/octo_shadow/octo_predictions.jsonl')
# Back on the laptop: copy it into the same octo_shadow/ folder, then
#   python src/octo_shadow.py merge logs/experiments/shadow/octo_shadow